In [ ]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import yaml

sys.path.append('/home/lishengping/projects/maxtext/MaxText')
# os.environ['HARDWARE'] = 'cpu'

import pyconfig
from layers import models
import max_utils
import jax
import orbax
import jax.numpy as jnp
from jax.sharding import Mesh
from flax.traverse_util import flatten_dict, unflatten_dict
from flax import linen as nn


# class DreamMiniXLE64T4Align(DreamMiniXLE64T4):
#     # 配置文件需要更改的几个地方：
#     base_output_directory = 'gs://newproject-1-llm_base_models_europe-west4'
#     run_name = 'test'
#     query_chunk_size = None # 如果传了这个参数，forward需要是query_chunk_size的整数倍
#     attention = 'dot_product_chunk'
#     # exp_class set your model class
#     per_device_batch_size = 1 # 可以根据测试的batch size定，设小一点主要是为了节省显存
#     max_target_length = 4096 # 可以根据测试的长度定，设小一点主要是为了节省显存
#     zero_loss = True
#      # 因为base.yml默认为空，必须写一个
#     scan_layers = False # 因为转模型的时候转的是scan_layers=False
#     record_internal_nn_metrics = 0
#     load_balance_loss_weight = None # dropless moe
#     megablox = False
#     bucket_logging_enabled = False
    # train_stage = 4
    
run_name = 'test'
os.makedirs(run_name, exist_ok=True) # 因为如果不存在会报错
config_name = '/home/lishengping/projects/maxtext/MaxText/configs/base.yml'
argv = [None, config_name]
config = pyconfig.initialize(argv)

2025-10-14 05:37:27.063685: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760420247.076776   21403 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760420247.080595   21403 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760420247.092122   21403 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760420247.092137   21403 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760420247.092139   21403 computation_placer.cc:177] computation placer alr

Updating keys from env and command line: []
Running Model: default
Updating keys from model: []
Attempting to initialize the jax distributed system...


2025-10-14 05:37:31.106889: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Jax distributed system initialized!


Updated exp model vars:
[EXP] adam_b1: 0.9
[EXP] adam_b2: 0.95
[EXP] adam_eps: 1e-08
[EXP] adam_weight_decay: 0.1
[EXP] async_checkpointing: True
[EXP] attention: dot_product_chunk
[EXP] base_emb_dim: 2048
[EXP] base_lr: 0.0004
[EXP] base_mlp_dim: 704
[EXP] base_num_decoder_layers: 36
[EXP] base_num_kv_heads: 32
[EXP] base_num_query_heads: 32
[EXP] base_output_directory: gs://newproject-1-llm_base_models_us-east5
[EXP] bucket_logging_enabled: False
[EXP] checkpoint_period: 100
[EXP] cosine_learning_rate_final_fraction: 0.035355339059327376
[EXP] data_shuffle_seed: 9876
[EXP] dataset_type: xm3.5mini
[EXP] dc_share_prepost_dw_hidden: True
[EXP] ddw_gen_chunk_size: None
[EXP] ddw_gen_pattern: q,k,v,m
[EXP] decay_method: cosine
[EXP] decoder_block: fusion
[EXP] dense_conn: True
[EXP] dynamic_dense_act_cls: gelu
[EXP] dynamic_dense_fix_last_layer: True
[EXP] dynamic_dense_hidden_round: True
[EXP] dynamic_dense_scale_dw: False
[EXP] dynamic_dense_type: q

In [2]:
# pip uninstall torch -y
# pip cache purge
# pip install torch --extra-index-url https://download.pytorch.org/whl/cpu
import torch
import numpy as np
import jax.numpy as jnp

# os.environ['HARDWARE'] = 'cpu'

import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


import json

p = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXLE64T40728/v3.5mini_moe_params_shape.json'
p = epath.Path(p)
with p.open('r') as f:
    shapedtype = json.load(f)
    
# load
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
axes = [1] * len(mesh_axes)
axes[2] = 4
devices = np.asarray(jax.devices()).reshape(axes)
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = jnp.bfloat16 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, shape in shapedtype.items():
    if not isinstance(k, tuple):
        k = tuple(k.split('/'))
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)    
    
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)
checkpoint_dir = 'gs://newproject-1-llm_base_models_us-east5/v3.5mini/DreamMiniXLE64T40728/checkpoints/178000/items'

ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args}
)

('params', 'decoder', 'compose_0', 'mudd_postnorm_0', 'scale') [2048]
('params', 'decoder', 'compose_0', 'mudd_prenorm_0', 'scale') [2048]
('params', 'decoder', 'compose_1', 'mudd_postnorm_1', 'scale') [2048]
('params', 'decoder', 'compose_1', 'mudd_prenorm_1', 'scale') [2048]
('params', 'decoder', 'compose_10', 'mudd_postnorm_10', 'scale') [2048]
('params', 'decoder', 'compose_10', 'mudd_prenorm_10', 'scale') [2048]
('params', 'decoder', 'compose_11', 'mudd_postnorm_11', 'scale') [2048]
('params', 'decoder', 'compose_11', 'mudd_prenorm_11', 'scale') [2048]
('params', 'decoder', 'compose_12', 'mudd_postnorm_12', 'scale') [2048]
('params', 'decoder', 'compose_12', 'mudd_prenorm_12', 'scale') [2048]
('params', 'decoder', 'compose_13', 'mudd_postnorm_13', 'scale') [2048]
('params', 'decoder', 'compose_13', 'mudd_prenorm_13', 'scale') [2048]
('params', 'decoder', 'compose_14', 'mudd_postnorm_14', 'scale') [2048]
('params', 'decoder', 'compose_14', 'mudd_prenorm_14', 'scale') [2048]
('param

I1014 05:37:41.399102   22748 google_auth_provider.cc:181] Running on GCE, using service account 626151558586-compute@developer.gserviceaccount.com


In [ ]:
def model_init(model, config, key):
  input_shape = (config.global_batch_size_to_load, config.max_target_length)
  params = model.init(
      {"params": key, "dropout": key, "aqt": key},
      jnp.ones(input_shape, dtype=jnp.int32),
      jnp.ones(input_shape, dtype=jnp.int32),
  )
  return params

quant = None
# devices_array = max_utils.create_device_mesh(config)
# mesh = Mesh(devices_array, config.mesh_axes)
Transformer = models.Transformer
jax_model = Transformer(config, mesh, quant=quant)

is_train = False
rng1, aqt_rng = jax.random.split(jax.random.key(9876))


In [3]:
import os
import time
import argparse
import socket
import random
from collections import defaultdict

# os.environ["JAX_PLATFORMS"] = "cpu"

import tensorflow as tf
import jax
import numpy as np


def _parse_function(example_proto):
    feature_desc = {key: tf.io.VarLenFeature(tf.int64) for key in task_features}
    example = tf.io.parse_single_example(example_proto, feature_desc)
    for name in list(example.keys()):
        t = example[name]
        if t.dtype == tf.int64:
            t = tf.cast(t, dtype=tf.int32)
        example[name] = tf.sparse.to_dense(t, default_value=0)[: seq_len]
        print(f'example[name]: {example[name]}')
    return example

task_features = {'input_ids': None}
train_seed = 1234
num_infeed_hosts = 1
shuffle_buffer_size = None
pad_id = 0
batch_size = 1
seq_len = 4097

p = 'gs://newproject-1-llm_base_models_us-east5/data/xiaomeng/v3.5mini/unigram_tfids0714/validation/R000.000000'
fname = [p]
tf.random.set_seed(train_seed)
ds = tf.data.Dataset.from_tensor_slices(fname)
ds = ds.apply(tf.data.TFRecordDataset)
ds = ds.shard(num_infeed_hosts, 0)
ds = ds.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
padded_shapes = {key: seq_len for key in task_features}
padding_values = {key: pad_id for key in task_features}
ds = ds.padded_batch(
    batch_size=np.prod(batch_size),
    padded_shapes=padded_shapes,
    padding_values=padding_values,
    drop_remainder=True,
)
ds_iter = ds.as_numpy_iterator()

example[name]: Tensor("strided_slice:0", shape=(None,), dtype=int32)


In [ ]:
jax_losses = []
length = 256
for i in range(0, 1):
    a = next(ds_iter)
    batch_indexes = torch.tensor([0])
    inputs = torch.from_numpy(a['input_ids'][:, :length]).long()
    labels = torch.from_numpy(a['input_ids'][:, 1:length+1]).long()
    input_pos = torch.arange(length).unsqueeze(0)

    batch_size = 1
    input_ids = jnp.array(inputs).reshape(batch_size, -1)
    data = {}
    data['inputs'] = input_ids
    data["inputs_position"] = jnp.arange(data['inputs'].shape[1]).reshape(batch_size, -1)
    data["inputs_segmentation"] = jnp.ones_like(data['inputs'])
    data["targets"] = jnp.array(labels).reshape(batch_size, -1)
    
    params = restored['params']['params']
    jax_logits, intermediate_outputs = jax_model.apply(
          {'params': params},
          data["inputs"],
          data["inputs_position"],
          decoder_segment_ids=data["inputs_segmentation"],
          enable_dropout=config.enable_dropout if is_train else False,
          rngs={"dropout": rng1, "params": aqt_rng},
          mutable="intermediates",
      )
    one_hot_targets = jax.nn.one_hot(data["targets"], config.vocab_size)
    jax_loss, _ = max_utils.cross_entropy_with_logits(jax_logits, one_hot_targets, 0.0)
    loss = round(jax_loss.mean().item(), 4)
    # print(f'jax mean loss: {jax_loss.mean()} shape: {jax_loss.shape}')
    print(f'loss: {loss}========================')
    jax_losses.append(loss)
    

/tmp/ipykernel_21403/173737024.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  inputs = torch.from_numpy(a['input_ids'][:, :length]).long()


num_kv_heads: 32
output: (1, 256, 32, 64)
sliding_window_size: 256 query_chunk_method:  eos_sum: None
global attn_mask: (1, 1, 256, 256) eos_sum: None
query_chunk_size is None
key_wise: False static_proj: False
key_wise: False static_proj: False
Running MoE sparse matmul implementation.
Enter permute sigmoid function......
num_kv_heads: 32
output: (1, 256, 32, 64)
sliding_window_size: 256 query_chunk_method:  eos_sum: None
global attn_mask: (1, 1, 256, 256) eos_sum: None
query_chunk_size is None
key_wise: False static_proj: False
key_wise: False static_proj: False
Running MoE sparse matmul implementation.
Enter permute sigmoid function......
num_kv_heads: 32
output: (1, 256, 32, 64)
sliding_window_size: 256 query_chunk_method:  eos_sum: None
global attn_mask: (1, 1, 256, 256) eos_sum: None
query_chunk_size is None
key_wise: False static_proj: False
key_wise: False static_proj: False
Running MoE sparse matmul implementation.
Enter permute sigmoid function......
num_kv_heads: 32
output: 